# Purpose

Explore PySpark and the JDBC connection functionality to read from operational databases.

In this notebook we will setup a PostgreSQL instance and populate it with the Pagila dataset. We will then connect to the database via a JDBC connector.

# Setup

## PostgreSQL

Firstly, let's install postgres in the this Colab instance.

In [78]:
!!sudo apt-get update
!sudo apt install postgresql postgresql-contrib

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
postgresql is already the newest version (14+238).
postgresql-contrib is already the newest version (14+238).
0 upgraded, 0 newly installed, 0 to remove and 39 not upgraded.


In [79]:
!service postgresql start

 * Starting PostgreSQL 14 database server
   ...done.


Create a user in Postgres ([stackoverflow](https://stackoverflow.com/questions/12720967/how-to-change-postgresql-user-password/12721020#12721020))


In [113]:
!sudo -u postgres psql -c "ALTER USER postgres PASSWORD 'test';"

ALTER ROLE


Store you database password in an environmental variable so that we need no type it in all the time (not advisable generally).

We'll use the notebook magic `%end`

In [105]:
%env PGPASSWORD=test

env: PGPASSWORD=test


## Pagila

Now, let's populate the PostgreSQL database with the Pagila data from the tutorial.

In [106]:
!git clone https://github.com/spatialedge-ai/pagila.git

fatal: destination path 'pagila' already exists and is not an empty directory.


In [107]:
!psql -h localhost -U postgres -c "create database pagila"

ERROR:  database "pagila" already exists


In [108]:
!psql -h localhost -U postgres -d pagila -f "pagila/pagila-schema.sql"

SET
SET
SET
SET
SET
 set_config 
------------
 
(1 row)

SET
SET
SET
SET
psql:pagila/pagila-schema.sql:29: ERROR:  type "mpaa_rating" already exists
ALTER TYPE
psql:pagila/pagila-schema.sql:39: ERROR:  type "year" already exists
ALTER DOMAIN
psql:pagila/pagila-schema.sql:56: ERROR:  function "_group_concat" already exists with same argument types
ALTER FUNCTION
psql:pagila/pagila-schema.sql:73: ERROR:  function "film_in_stock" already exists with same argument types
ALTER FUNCTION
psql:pagila/pagila-schema.sql:90: ERROR:  function "film_not_in_stock" already exists with same argument types
ALTER FUNCTION
psql:pagila/pagila-schema.sql:135: ERROR:  function "get_customer_balance" already exists with same argument types
ALTER FUNCTION
psql:pagila/pagila-schema.sql:157: ERROR:  function "inventory_held_by_customer" already exists with same argument types
ALTER FUNCTION
psql:pagila/pagila-schema.sql:194: ERROR:  function "inventory_in_stock" already exists with same argument types
ALTER FUN

In [109]:
!psql -h localhost -U postgres -d pagila -f "pagila/pagila-data.sql"

SET
SET
SET
SET
SET
 set_config 
------------
 
(1 row)

SET
SET
SET
SET
psql:pagila/pagila-data.sql:224: ERROR:  duplicate key value violates unique constraint "actor_pkey"
DETAIL:  Key (actor_id)=(1) already exists.
CONTEXT:  COPY actor, line 1
psql:pagila/pagila-data.sql:341: ERROR:  duplicate key value violates unique constraint "country_pkey"
DETAIL:  Key (country_id)=(1) already exists.
CONTEXT:  COPY country, line 1
psql:pagila/pagila-data.sql:949: ERROR:  duplicate key value violates unique constraint "city_pkey"
DETAIL:  Key (city_id)=(1) already exists.
CONTEXT:  COPY city, line 1
psql:pagila/pagila-data.sql:1560: ERROR:  duplicate key value violates unique constraint "address_pkey"
DETAIL:  Key (address_id)=(1) already exists.
CONTEXT:  COPY address, line 1
psql:pagila/pagila-data.sql:1584: ERROR:  duplicate key value violates unique constraint "category_pkey"
DETAIL:  Key (category_id)=(1) already exists.
CONTEXT:  COPY category, line 1
psql:pagila/pagila-data.sql:1594: ERR

## PySpark Setup

Now, let's download what is necessary for initiating jdbc connections, as well as what is required to run PySpark itself.

In [110]:
# https://stackoverflow.com/questions/34948296/using-pyspark-to-connect-to-postgresql
!wget https://jdbc.postgresql.org/download/postgresql-42.5.0.jar

--2025-10-03 20:09:32--  https://jdbc.postgresql.org/download/postgresql-42.5.0.jar
Resolving jdbc.postgresql.org (jdbc.postgresql.org)... 72.32.157.228, 2001:4800:3e1:1::228
Connecting to jdbc.postgresql.org (jdbc.postgresql.org)|72.32.157.228|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1046274 (1022K) [application/java-archive]
Saving to: ‘postgresql-42.5.0.jar.3’

postgresql-42.5.0.j 100%[===================>]   1022K  1.25MB/s    in 0.8s    

2025-10-03 20:09:34 (1.25 MB/s) - ‘postgresql-42.5.0.jar.3’ saved [1046274/1046274]



In [97]:
import os
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
import numpy as np

%config Completer.use_jedi = False

SPARKVERSION='3.2.1'
HADOOPVERSION='3.2'
pwd=os.getcwd()

os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-8-openjdk-amd64"
os.environ["SPARK_HOME"] = f"{pwd}/spark-{SPARKVERSION}-bin-hadoop{HADOOPVERSION}"

# print(os.environ['SPARK_HOME'])


In [60]:
!sudo apt-get install openjdk-8-jdk-headless -qq > /dev/null
!wget https://archive.apache.org/dist/spark/spark-{SPARKVERSION}/spark-{SPARKVERSION}-bin-hadoop{HADOOPVERSION}.tgz
!tar xf spark-{SPARKVERSION}-bin-hadoop{HADOOPVERSION}.tgz

--2025-10-03 18:06:52--  https://archive.apache.org/dist/spark/spark-3.2.1/spark-3.2.1-bin-hadoop3.2.tgz
Resolving archive.apache.org (archive.apache.org)... 65.108.204.189, 2a01:4f9:1a:a084::2
Connecting to archive.apache.org (archive.apache.org)|65.108.204.189|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 300971569 (287M) [application/x-gzip]
Saving to: ‘spark-3.2.1-bin-hadoop3.2.tgz.2’

spark-3.2.1-bin-had 100%[===================>] 287.03M   305KB/s    in 33m 2s  

2025-10-03 18:39:55 (148 KB/s) - ‘spark-3.2.1-bin-hadoop3.2.tgz.2’ saved [300971569/300971569]



In [90]:
!cp postgresql-42.5.0.jar spark-{SPARKVERSION}-bin-hadoop{HADOOPVERSION}/jars

In [89]:
!pip install findspark

In [92]:
import findspark
findspark.init()
findspark.find()

# get a spark session
from pyspark.sql import SparkSession

# Correct JAR path
jdbc_driver_path = "/content/postgresql-42.5.0.jar"

spark = SparkSession.builder \
    .config("spark.jars", jdbc_driver_path) \
    .config("spark.driver.extraClassPath", jdbc_driver_path) \
    .getOrCreate()

print("Loaded JAR:", spark.conf.get("spark.jars"))

%env PYARROW_IGNORE_TIMEZONE=1

Loaded JAR: /content/postgresql-42.5.0.jar
env: PYARROW_IGNORE_TIMEZONE=1


# Questions

### Question 1

Using a PySpark dataframe, print the schema of customer table in the pagila PostgreSQL database by utilising a JDBC connection.

In [ ]:
# pyspark code
jdbc_url = "jdbc:postgresql://<HOST>:<PORT>/<DBNAME>"

connection_props = {
    "user": "YOUR_POSTGRES_USERNAME",
    "password": "YOUR_POSTGRES_PASSWORD",
    "driver": "org.postgresql.Driver"
}

# Load the 'customer' table into a PySpark DataFrame
customer_df = spark.read.jdbc(
    url=jdbc_url,
    table="customer",
    properties=connection_props
)

# Print schema of customer table
customer_df.printSchema()

### Question 2

Use the Spark SQL API to query the customer table, compute the number of unique email addresses in that table and print the result in the notebook.

In [ ]:
# pyspark code
from pyspark.sql import SparkSession

# Initialize Spark session
spark = SparkSession.builder \
    .appName("UniqueEmailsQuery") \
    .getOrCreate()

# JDBC connection details
jdbc_url = "jdbc:postgresql://<HOST>:<PORT>/pagila"
connection_properties = {
    "user": "<USERNAME>",
    "password": "<PASSWORD>",
    "driver": "org.postgresql.Driver"
}

# Load customer table into DataFrame
customer_df = spark.read.jdbc(
    url=jdbc_url,
    table="customer",
    properties=connection_properties
)

# Register DataFrame as a temporary SQL view
customer_df.createOrReplaceTempView("customer")

# Run SQL query to count distinct email addresses
result = spark.sql("SELECT COUNT(DISTINCT email) AS unique_emails FROM customer")

# Show result
result.show()

### Question 3

Repeat this calculation using only the Dataframe API and print the result.

In [ ]:
# pyspark code
from pyspark.sql import SparkSession

# Initialize Spark session
spark = SparkSession.builder \
    .appName("UniqueEmailsDF") \
    .getOrCreate()

# JDBC connection details
jdbc_url = "jdbc:postgresql://<HOST>:<PORT>/pagila"
connection_properties = {
    "user": "<USERNAME>",
    "password": "<PASSWORD>",
    "driver": "org.postgresql.Driver"
}

# Load customer table into DataFrame
customer_df = spark.read.jdbc(
    url=jdbc_url,
    table="customer",
    properties=connection_properties
)

# Count distinct email addresses using DataFrame API
unique_email_count = customer_df.select("email").distinct().count()

# Print result
print(f"Number of unique email addresses: {unique_email_count}")

### Question 4

How many partitions are present in the dataframe resulting from Question 3 (additionally provide the code necessary to determine that)

In [ ]:

# Check number of partitions in the DataFrame
num_partitions = customer_df.rdd.getNumPartitions()

# Print result
print(f"Number of partitions: {num_partitions}")


### Question 5

Compute the min and max of customer.create_date and print the result (once more using the Spark DataFrame API and not the Spark SQL API).

In [ ]:
from pyspark.sql.functions import min, max

# Compute min and max of create_date
min_max_df = customer_df.agg(
    min("create_date").alias("min_create_date"),
    max("create_date").alias("max_create_date")
)

# Show the result
min_max_df.show()

### Question 6.1

Determine which first names occur more than once:

1. using the Spark SQL API (printing the result)

In [ ]:
# Register the customer DataFrame as a temporary view
customer_df.createOrReplaceTempView("customer_view")

# Use Spark SQL to find first names occurring more than once
duplicate_first_names_df = spark.sql("""
    SELECT first_name, COUNT(*) AS name_count
    FROM customer_view
    GROUP BY first_name
    HAVING COUNT(*) > 1
    ORDER BY name_count DESC
""")

# Print the result
duplicate_first_names_df.show()

### Question 6.2

  2. using the Spark Dataframe API (printing the result once more).

In [ ]:
from pyspark.sql.functions import col, count

# Group by first_name and filter those with count > 1
duplicate_names_df = customer_df.groupBy("first_name") \
    .agg(count("*").alias("name_count")) \
    .filter(col("name_count") > 1)

# Show the result
duplicate_names_df.show()


### Question 7

Port the PostgreSQL below to the PySpark DataFrame API and execute the query within Spark (not directly on PostgreSQL):

```
SELECT
   staff.first_name
   ,staff.last_name
   ,SUM(payment.amount)
 FROM payment
   INNER JOIN staff ON payment.staff_id = staff.staff_id
 WHERE payment.payment_date BETWEEN '2007-01-01' AND '2007-02-01'
 GROUP BY
   staff.last_name
   ,staff.first_name
 ORDER BY SUM(payment.amount)
 ;
```

In [ ]:
# Load payment table
payment_df = spark.read.jdbc(
    url=jdbc_url,
    table="payment",
    properties=connection_props
)

# Load staff table
staff_df = spark.read.jdbc(
    url=jdbc_url,
    table="staff",
    properties=connection_props
)

from pyspark.sql.functions import sum, col

# Filter payment dates between '2007-01-01' and '2007-02-01'
payment_filtered = payment_df.filter(
    (col("payment_date") >= "2007-01-01") & (col("payment_date") <= "2007-02-01")
)

# Join with staff table
joined_df = payment_filtered.join(
    staff_df,
    payment_filtered.staff_id == staff_df.staff_id,
    how="inner"
)

# Group by staff first_name and last_name and sum the payment amount
result_df = joined_df.groupBy(
    staff_df.first_name, staff_df.last_name
).agg(
    sum("amount").alias("total_amount")
).orderBy(
    col("total_amount")
)

# Show the result
result_df.show()


### Question 8

Are you currently executing commands on a driver node, or a worker? Provide the code you ran to determine that.

In [ ]:

import socket
from pyspark.sql import SparkSession

# Initialize Spark session
spark = SparkSession.builder.getOrCreate()

# Get the hostname of the node executing this code
hostname = socket.gethostname()

print(f"Code is executing on node: {hostname}")
